In [ ]:
from __future__ import annotations

import json
import logging
import os
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Optional

import openai
from openai import OpenAI

In [ ]:
@dataclass
class ExperimentConfig:
    root_dir: str
    output_dir: str
    model: str = "gpt-5.4-mini"
    reasoning_effort: str = "low"
    max_output_tokens: int = 4000
    timeout_seconds: float = 120.0
    max_retries: int = 2
    overwrite_jsonl: bool = True
    prompt_version: str = "zero_shot_v1"

In [ ]:
ROOT_DIR = "../../shared/input/extracted_snippets"
OUTPUT_DIR = "../results"

BASE_INSTRUCTIONS = ""

USER_PROMPT_TEMPLATE = """
This is a library migration task, not a general rewrite task.
Replace pandas-based operations with the closest correct polars equivalents.

Requirements:
1. Preserve the original functionality and data-processing intent as much as possible.
2. Only make changes that are necessary for the pandas-to-polars migration.
3. Do not refactor, optimize, reformat, or modify unrelated code.
4. Do not rename functions, classes, or variables unless necessary.
5. Preserve the original structure as much as possible while completing the migration correctly.
6. Preserve output columns, schema, and overall data behavior whenever possible.
7. Preserve filtering, aggregation, join, and ordering behavior when relevant.
8. Handle pandas index-related logic carefully. Do not silently remove behavior that depends on index semantics.
9. Use valid polars syntax, imports, and APIs. Do not invent nonexistent polars functionality.
10. When a direct one-to-one replacement is not possible, use the closest correct polars equivalent while preserving behavior as much as possible.

Return only the full migrated code.
Do not include explanations.
Do not use Markdown fences.

Code:
{code}
""".strip()

config = ExperimentConfig(
    root_dir=ROOT_DIR,
    output_dir=OUTPUT_DIR,
    model="gpt-5.4-mini",
    reasoning_effort="low",
    max_output_tokens=4000,
    timeout_seconds=120.0,
    max_retries=2,
    overwrite_jsonl=True,
    prompt_version="zero_shot_v1",
)

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY in the environment before running generation.")

client = OpenAI(
    timeout=config.timeout_seconds,
    max_retries=config.max_retries,
)


In [ ]:
def find_before_files(root_dir: str) -> list[Path]:
    """
    递归查找所有严格命名为 before.py 的文件。
    """
    root = Path(root_dir)
    return sorted(root.rglob("before.py"))


def build_input_text(prompt_template: str, before_code: str) -> str:
    return prompt_template.format(code=before_code)


def extract_path_metadata(before_file: Path, root_dir: str) -> dict:
    """
    针对 extracted_snippets 的目录结构提取信息。

    预期结构类似：
    extracted_snippets/
        repo/
            sha/
                file_path_dir/
                    before.py
                    after.py

    例如：
    extracted_snippets/YuriNakayama__cate/f11c29.../cate__dataset.py/before.py
    """
    root = Path(root_dir)
    rel_path = before_file.relative_to(root)
    parts = rel_path.parts

    repo = parts[0] if len(parts) >= 1 else None
    sha = parts[1] if len(parts) >= 2 else None

    parent_parts = parts[2:-1] if len(parts) > 3 else []
    original_file_path = "/".join(parent_parts) if parent_parts else None

    pair_id = None
    if repo and sha and original_file_path:
        pair_id = f"{repo}__{sha}__{original_file_path}"
    elif repo and sha:
        pair_id = f"{repo}__{sha}"

    return {
        "pair_id": pair_id,
        "repo": repo,
        "sha": sha,
        "relative_path": str(rel_path),
        "original_file_path": original_file_path,
        "file_name": before_file.name,
    }


def make_output_path(before_file: Path, root_dir: str, output_dir: str) -> Path:
    """
    保持和输入目录相同的层级结构，
    只是把 before.py 保存成 generated_polars.py
    """
    root = Path(root_dir)
    out_root = Path(output_dir)

    rel_path = before_file.relative_to(root)
    new_name = f"{before_file.stem}_after_generated.py"
    return out_root / rel_path.parent / new_name


def append_jsonl(record: dict, jsonl_path: Path) -> None:
    """
    向 jsonl 文件追加一条记录。
    """
    jsonl_path.parent.mkdir(parents=True, exist_ok=True)
    with jsonl_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

In [ ]:
def generate_from_before_file(
    before_file: Path,
    prompt_template: str,
    base_instructions: str,
    model: str,
    reasoning_effort: str,
    max_output_tokens: int,
    root_dir: str,
) -> dict:
    start = time.perf_counter()
    metadata = extract_path_metadata(before_file, root_dir)

    try:
        before_code = before_file.read_text(encoding="utf-8")
        print("About to call OpenAI API...")

        response = client.responses.create(
            model=model,
            reasoning={"effort": reasoning_effort},
            instructions=base_instructions,
            input=build_input_text(prompt_template=prompt_template, before_code=before_code),
            max_output_tokens=max_output_tokens,
        )
        print("OpenAI API call finished.")

        latency = time.perf_counter() - start

        return {
            **metadata,
            "before_file": str(before_file),
            "success": True,
            "generated_code": response.output_text,
            "error_type": None,
            "error_message": None,
            "latency_seconds": round(latency, 3),
            "model": model,
            "prompt_version": config.prompt_version,
        }

    except openai.RateLimitError as e:
        latency = time.perf_counter() - start
        return {
            **metadata,
            "before_file": str(before_file),
            "success": False,
            "generated_code": None,
            "error_type": type(e).__name__,
            "error_message": str(e),
            "latency_seconds": round(latency, 3),
            "model": model,
            "prompt_version": config.prompt_version,
        }

    except openai.APIConnectionError as e:
        latency = time.perf_counter() - start
        return {
            **metadata,
            "before_file": str(before_file),
            "success": False,
            "generated_code": None,
            "error_type": type(e).__name__,
            "error_message": str(e),
            "latency_seconds": round(latency, 3),
            "model": model,
            "prompt_version": config.prompt_version,
        }

    except openai.APIStatusError as e:
        latency = time.perf_counter() - start
        return {
            **metadata,
            "before_file": str(before_file),
            "success": False,
            "generated_code": None,
            "error_type": type(e).__name__,
            "error_message": str(e),
            "latency_seconds": round(latency, 3),
            "model": model,
            "prompt_version": config.prompt_version,
        }

    except openai.APIError as e:
        latency = time.perf_counter() - start
        return {
            **metadata,
            "before_file": str(before_file),
            "success": False,
            "generated_code": None,
            "error_type": type(e).__name__,
            "error_message": str(e),
            "latency_seconds": round(latency, 3),
            "model": model,
            "prompt_version": config.prompt_version,
        }

    except Exception as e:
        latency = time.perf_counter() - start
        return {
            **metadata,
            "before_file": str(before_file),
            "success": False,
            "generated_code": None,
            "error_type": type(e).__name__,
            "error_message": str(e),
            "latency_seconds": round(latency, 3),
            "model": model,
            "prompt_version": config.prompt_version,
        }

In [ ]:
def save_single_result(result: dict, root_dir: str, output_dir: str) -> Optional[Path]:
    """
    成功时保存 generated_polars.py
    失败时保存 generation_error.txt
    """
    before_file = Path(result["before_file"])
    output_path = make_output_path(before_file, root_dir, output_dir)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    if result["success"]:
        output_path.write_text(result["generated_code"] or "", encoding="utf-8")
        return output_path
    else:
        error_path = output_path.parent / "generation_error.txt"
        error_text = (
            f"ERROR\n"
            f"before_file: {result['before_file']}\n"
            f"error_type: {result['error_type']}\n"
            f"error_message: {result['error_message']}\n"
        )
        error_path.write_text(error_text, encoding="utf-8")
        return error_path

In [ ]:
def find_before_files(root_dir: str, keyword: str = "before") -> list[Path]:
    root = Path(root_dir)
    all_files = sorted(
        p for p in root.rglob("*.py")
        if keyword.lower() in p.name.lower()
    )
    result = []
    skipped = []
    for f in all_files:
        try:
            first_line = f.read_text(encoding="utf-8").split("\n", 1)[0].strip()
            if first_line.startswith("# DISABLED"):
                skipped.append(f.name)
                continue
        except Exception:
            pass
        result.append(f)
    print(f"Skipped {len(skipped)} DISABLED files.")
    return result

before_files = find_before_files(config.root_dir, keyword="before")

print(f"Found {len(before_files)} files to process.")
for p in before_files[:10]:
    print(p)

In [ ]:
# test_file = Path("../../shared/input/extracted_snippets/YuriNakayama__cate/f11c29aece42e2a62be6379ee2eeb7175bb7ad66/cate__dataset.py/f11c29ae_dataset_to_pandas_before.py")

# test_result = generate_from_before_file(
#     before_file=test_file,
#     prompt_template=USER_PROMPT_TEMPLATE,
#     base_instructions=BASE_INSTRUCTIONS,
#     model=config.model,
#     reasoning_effort=config.reasoning_effort,
#     max_output_tokens=config.max_output_tokens,
#     root_dir=config.root_dir,
# )

# test_save_path = save_single_result(
#     result=test_result,
#     root_dir=config.root_dir,
#     output_dir=config.output_dir,
# )

# print("Test success:", test_result["success"])
# print("Saved to:", test_save_path)
# print("Pair ID:", test_result["pair_id"])
# print("Repo:", test_result["repo"])
# print("SHA:", test_result["sha"])

In [ ]:
results_jsonl = Path(config.output_dir) / "results.jsonl"

if config.overwrite_jsonl and results_jsonl.exists():
    results_jsonl.unlink()

all_results = []

for i, before_file in enumerate(before_files, start=1):
    print(f"[{i}/{len(before_files)}] Processing: {before_file}")

    result = generate_from_before_file(
        before_file=before_file,
        prompt_template=USER_PROMPT_TEMPLATE,
        base_instructions=BASE_INSTRUCTIONS,
        model=config.model,
        reasoning_effort=config.reasoning_effort,
        max_output_tokens=config.max_output_tokens,
        root_dir=config.root_dir,
    )

    save_path = save_single_result(
        result=result,
        root_dir=config.root_dir,
        output_dir=config.output_dir,
    )

    result["saved_path"] = str(save_path) if save_path else None

    append_jsonl(result, results_jsonl)
    all_results.append(result)

print("Done.")
print(f"Results JSONL saved to: {results_jsonl}")

In [ ]:
success_count = sum(r["success"] for r in all_results)
fail_count = len(all_results) - success_count

print("Total:", len(all_results))
print("Success:", success_count)
print("Fail:", fail_count)